# Summarizer Evaluation

Calls `/summarize` for each fixture and measures:
- **ROUGE-1 F** between the returned summary and the original text
- **Keyword hit rate** for expected words in headline and summary
- **Latency** per call

**Prerequisite:** NLP service running at `http://localhost:8001` (`/readyz` returns 200).

In [3]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, rouge1_f, keyword_hit_rate, NLP_BASE_URL, HEADERS

cases = load_fixture('summarize_cases.json')
print(f'Loaded {len(cases)} test cases')

Loaded 5 test cases


In [8]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
print(f"Loaded API Key: '{HEADERS}'")
# This will tell us the exact HTTP code and the raw text error message
print(f"Status Code: {r.status_code}")
print(f"Response Body: {r.text}")

assert r.status_code == 200, 'Service not ready — start the NLP service first'

Loaded API Key: '{'X-API-Key': '191474acfe2e7fe38056fa5bc4635c3c4a413c7c9451b7fa36c40842060822c3'}'
Status Code: 503
Response Body: {"status":"warming","missing":["dedup","summarize"]}


AssertionError: Service not ready — start the NLP service first

In [9]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/summarize',
        json={
            'article_id': case['article_id'],
            'text': case['text'],
            'raw_headline': case['raw_headline'],
        }, 
        headers=HEADERS
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, f"{case['id']}: HTTP {resp.status_code} — {resp.text}"
    data = resp.json()

    r1 = rouge1_f(case['text'], data['summary'])
    hl_hit = keyword_hit_rate(data['headline'], case.get('expected_headline_keywords', []))
    sum_hit = keyword_hit_rate(data['summary'], case.get('expected_summary_keywords', []))
    pass_fail = '✅' if r1 >= case.get('min_rouge1', 0.2) else '❌'

    results.append({
        'id': case['id'],
        'description': case['description'],
        'rouge1': r1,
        'headline_hit': hl_hit,
        'summary_keyword_hit': sum_hit,
        'latency_s': latency,
        'headline': data['headline'],
        'summary': data['summary'],
        'pass': r1 >= case.get('min_rouge1', 0.2),
    })

    print(f"{pass_fail} [{case['id']}] ROUGE-1={r1:.3f}  hl_hit={hl_hit:.2f}  sum_hit={sum_hit:.2f}  {latency:.1f}s")
    print(f"   headline : {data['headline']}")
    print(f"   summary  : {data['summary'][:120]}...")
    print()

✅ [sum-001] ROUGE-1=0.580  hl_hit=1.00  sum_hit=0.67  93.6s
   headline : El Ayuntamiento de Madrid amplía el carril bici de la Gran Vía
   summary  : El Ayuntamiento de Madrid aprobará la ampliación del carril bici en la Gran Vía, extendiéndose desde la plaza de España ...

✅ [sum-002] ROUGE-1=0.698  hl_hit=1.00  sum_hit=1.00  17.6s
   headline : Estación de Atocha estrena nuevo aparcamiento cubierto para bicicletas
   summary  : La estación de Atocha ha inaugurado un aparcamiento cubierto con capacidad para 200 bicicletas. La instalación cuenta ad...

✅ [sum-003] ROUGE-1=0.277  hl_hit=1.00  sum_hit=0.00  59.0s
   headline : Barcelona aprueba su plan de movilidad sostenible para el periodo 2026-2030
   summary  : El Ayuntamiento de Barcelona ha aprobado un plan de movilidad sostenible con una inversión de 1.200 millones de euros. E...

✅ [sum-004] ROUGE-1=0.577  hl_hit=0.33  sum_hit=0.67  57.2s
   headline : Un ciclista resulta herido tras ser golpeado por un vehículo
   summary  : Un

In [10]:
passing = [r for r in results if r['pass']]
avg_rouge1 = sum(r['rouge1'] for r in results) / len(results)
avg_hl_hit = sum(r['headline_hit'] for r in results) / len(results)
avg_lat = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('SUMMARIZER', {
    'Cases': len(results),
    'Passing (ROUGE-1 ≥ threshold)': f"{len(passing)}/{len(results)}",
    'Average ROUGE-1 F': avg_rouge1,
    'Target ROUGE-1 F': 0.20,
    'Average headline keyword hit': avg_hl_hit,
    'Average latency (s)': avg_lat,
})


  SUMMARIZER
  Cases                               5
  Passing (ROUGE-1 ≥ threshold)       5/5
  Average ROUGE-1 F                   0.516
  Target ROUGE-1 F                    0.200
  Average headline keyword hit        0.867
  Average latency (s)                 56.328



## Tuning Guide

| Symptom | Lever | Where |
|---------|-------|-------|
| Summary too long / unfocused | Lower `max_sentences` (default 3) in the request or reduce the extractive pre-step | `summarizer/service.py: _EXTRACTIVE_WORD_THRESHOLD` |
| Extractive pre-step fires on short articles | Raise `_EXTRACTIVE_WORD_THRESHOLD` (currently 500) | `nlp/summarizer/service.py` |
| LLM produces hallucinated facts | Check Ollama model; switch `OLLAMA_MODEL` env var to a larger model | `docker-compose.yml` |
| Timeout errors | Raise `OLLAMA_TIMEOUT` env var (default 30 s) | `docker-compose.yml` |
| Headline too generic | Improve the prompt in `ollama_client.py` — add more specificity constraints | `nlp/summarizer/ollama_client.py` |